# Flare Watch Nigeria: pipeline walkthrough

This notebook shows how the processed data was built, step by step. It reads the files that
`scripts/` and `models/train_anomaly.py` produced, so run the pipeline first:

```
python scripts/download_data.py
python scripts/build_sites.py
python scripts/build_timeseries.py
python scripts/estimate_co2e.py
python models/train_anomaly.py
```

Every source file is public. The raw downloads are cached in `data/raw/` and are not committed.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
PROCESSED = ROOT / 'data' / 'processed'
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
print('project root:', ROOT)

## 1. The site registry

`scripts/build_sites.py` streams the World Bank per-location workbook, keeps only Nigerian rows,
assigns each site to a state using point-in-polygon against geoBoundaries ADM1 polygons, and finds
the nearest named settlement from OpenStreetMap.

In [ ]:
sites = pd.read_csv(PROCESSED / 'sites.csv')
print(f'{len(sites):,} flare sites, {sites["first_year"].min()} to {sites["last_year"].max()}')
print()
sites.groupby('state', dropna=False)['total_bcm'].sum().sort_values(ascending=False).head(10)

In [ ]:
sites.nlargest(12, 'total_bcm')[['id', 'state', 'operator', 'field_name', 'first_year', 'last_year', 'total_bcm', 'peak_bcm', 'nearest_settlement', 'nearest_settlement_km']]

Operator and field name come straight from the World Bank metadata. They are shown for context and
are never used to group, match or attribute anything.

## 2. The annual panel

`scripts/build_timeseries.py` expands every site across every year and derives the continuity,
persistence and comparison features the anomaly model consumes.

In [ ]:
panel = pd.read_csv(PROCESSED / 'site_year_panel.csv')
print(f'{len(panel):,} site-years = {panel["id"].nunique():,} sites x {panel["year"].nunique()} years')
print(f'active {panel["active"].sum():,}   dark {len(panel) - panel["active"].sum():,}')
print(f'relights {panel["relight"].sum():,}   shutdowns {panel["shutdown"].sum():,}   spikes {panel["volume_spike"].sum():,}')
panel.head()

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4))
yearly = panel.groupby('year')['BCM'].sum()
axes[0].bar(yearly.index, yearly.values, color='#2563eb')
axes[0].set_title('Nigeria flared gas by year')
axes[0].set_xlabel('year')
axes[0].set_ylabel('billion cubic metres')
axes[1].hist(panel.loc[panel['detection_freq'].notna(), 'detection_freq'], bins=30, color='#ef4444')
axes[1].set_title('Detection frequency (share of clear-sky passes)')
axes[1].set_xlabel('detection frequency')
plt.tight_layout()

## 3. Carbon and energy

The emission factor is 2.57 kg CO2e per cubic metre of gas flared, which is what the World Bank's
own global figures imply: 148 bcm paired with 381 Mt CO2e in 2023.

In [ ]:
co2e = pd.read_csv(PROCESSED / 'site_year_co2e.csv')
summary = pd.read_json(PROCESSED / 'national_summary.json', typ='series')
national = co2e.groupby('year').agg(bcm=('BCM', 'sum'), co2e_mt=('co2e_tonnes', lambda s: s.sum() / 1e6), homes_m=('homes_equivalent', lambda s: s.sum() / 1e6))
national.round(2)

## 4. Anomalies

A single Isolation Forest is fitted on every flaring site-year, and five deterministic rules catch
the behaviours regulators care about: relights, volume spikes, extended shutdowns, persistent flares
and new sites.

In [ ]:
anomalies = pd.read_csv(PROCESSED / 'anomalies.csv')
print(f'{len(anomalies):,} flagged site-years across {anomalies["id"].nunique():,} sites')
anomalies['flag_source'].value_counts()

In [ ]:
sample = anomalies[anomalies['rule_tags'].notna()].iloc[0]
print(f'site {sample["id"]} in {int(sample["year"])}, {sample["state"]}')
print('rules :', sample['rule_tags'])
print('model :', sample['shap_explanation'])

## 5. The worst flagged sites


In [ ]:
risk = pd.read_csv(PROCESSED / 'site_risk.csv')
risk.head(15)[['id', 'state', 'operator', 'field_name', 'flags', 'worst_score', 'latest_flag_year', 'total_bcm', 'nearest_settlement', 'nearest_settlement_km']]

In [ ]:
top = risk.nlargest(20, 'total_bcm')
figure, axes = plt.subplots(1, 2, figsize=(13, 5))
scatter = axes[0].scatter(top['lon'], top['lat'], s=top['total_bcm'] * 90, c=top['worst_score'], cmap='YlOrRd', alpha=0.75, edgecolor='white')
figure.colorbar(scatter, ax=axes[0], label='highest anomaly score')
axes[0].set_title('20 largest flare sites, shaded by anomaly score')
axes[0].set_xlabel('longitude')
axes[0].set_ylabel('latitude')
counts = anomalies['rule_tags'].dropna().str.count(',') + 1
axes[1].hist(counts, bins=range(1, counts.max() + 2), color='#2563eb', align='left')
axes[1].set_title('How many rules fired per flagged year')
axes[1].set_xlabel('rules fired')
axes[1].set_ylabel('flagged site-years')
plt.tight_layout()

## What this does not claim

Flare volumes are satellite-derived estimates calibrated against country-level reporting, not meter
readings. A year with no detection means VIIRS saw no flare, which usually means no flaring, but can
also mean cloud cover or a plume below the detection threshold. Absence of evidence is not evidence of
absence, and a shutdown flag is a signal to check, not a finding of compliance.